# FLUKE Sentiment Analysis with OpenAI o3-2025-04-16 Reasoning Model

This notebook evaluates sentiment analysis robustness using OpenAI's o3-2025-04-16 reasoning model with FLUKE linguistic modifications.

In [ ]:
from datasets import load_dataset
import dspy
import openai
import os
import re
import pandas as pd
import json
import random
from dotenv import load_dotenv
import glob
from scipy import stats
import time
from tqdm import tqdm

In [ ]:
load_dotenv()

In [ ]:
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

## Model Configuration

Configure o3-2025-04-16 reasoning model. Note: o3 models don't support temperature or system messages, and have limited parameters.

In [ ]:
# Available o3 and o1 reasoning models
REASONING_MODELS = {
    'o3-2025-04-16': 'openai/o3-2025-04-16',
    'o1-preview': 'openai/o1-preview',
    'o1-mini': 'openai/o1-mini',
    'o1': 'openai/o1',
}

# Select model to use - defaulting to latest o3 model
MODEL_NAME = 'o3-2025-04-16'  # Change this to test different models
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Using model: {MODEL_NAME} ({MODEL_ID})")

In [ ]:
# Configure DSPy with o3 model
# Note: o3 models, like o1, don't support temperature or max_tokens parameters
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

In [ ]:
# Load dataset
ds = load_dataset('stanfordnlp/sst2')['validation']
print(f"Dataset size: {len(ds)}")

In [ ]:
def remove_space(text):
    """Clean up spacing and formatting in dialogue text."""
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        # Remove multiple spaces
        cleaned = ' '.join(line.split())
        
        # Fix spacing around punctuation
        cleaned = re.sub(r'\s+([.,!?:;])', r'\1', cleaned)
        cleaned = re.sub(r'([.,!?:;])\s+', r'\1 ', cleaned)
        
        # Fix contractions
        cleaned = re.sub(r'\s*\'\s*s\b', "'s", cleaned)
        cleaned = re.sub(r'\s*n\s*\'\s*t\b', "n't", cleaned)
        cleaned = re.sub(r'\s*\'\s*ve\b', "'ve", cleaned)
        cleaned = re.sub(r'\s*\'\s*re\b', "'re", cleaned)
        cleaned = re.sub(r'\s*\'\s*ll\b', "'ll", cleaned)
        cleaned = re.sub(r'\s*\'\s*d\b', "'d", cleaned)
        cleaned = re.sub(r'\s*\'\s*m\b', "'m", cleaned)
        
        # Fix spaces around parentheses
        cleaned = re.sub(r'\(\s+', '(', cleaned)
        cleaned = re.sub(r'\s+\)', ')', cleaned)
        
        # Remove leading/trailing whitespace
        cleaned = cleaned.strip()
        cleaned_lines.append(cleaned)
        
    return '\n'.join(cleaned_lines)

In [ ]:
examples = [
    dspy.Example({ 
                  "text": remove_space(r["sentence"]), 
                  "label": r["label"]}
                  ).with_inputs("text") 
    for r in ds
]

In [ ]:
example = examples[835]
for k, v in example.items():
    print(f"\n{k.upper()}:\n")
    print(v)

In [ ]:
def extract_prediction(text):
    """Extract prediction from o3 model output."""
    # Look for explicit answers first
    patterns = [
        r'Answer:\s*([01])',
        r'Final answer:\s*([01])',
        r'Label:\s*([01])',
        r'Prediction:\s*([01])',
        r'Classification:\s*([01])',
        r'\b([01])\b'
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1]
    
    return ""

In [ ]:
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    parsed_answer = extract_prediction(pred)
    return parsed_answer == str(true.label)

# Evaluate the original test set

In [ ]:
from dspy.evaluate import Evaluate

## Sentiment Classification with o3 Model

o3 models automatically use advanced chain-of-thought reasoning.

In [ ]:
class O3Sentiment(dspy.Signature):
    """Classify sentiment of the given text. Think step by step and analyze the emotional tone, word choice, and overall sentiment. Answer with 1 for positive sentiment, 0 for negative sentiment."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

In [ ]:
class O3SentimentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Sentiment)

    def forward(self, text):
        return self.prog(text=text)

In [ ]:
o3_sentiment = O3SentimentModule()

In [ ]:
# Test with a single example
example = examples[835]
print(f"Text: {example.text}")
print(f"True Label: {example.label}")

pred = o3_sentiment(text=example.text)
print(f"\nPrediction: {pred}")
print(f"\nCorrect: {eval_metric(example, pred)}")

In [ ]:
eval_metric(example, pred)

## Evaluate Original Test Set

In [ ]:
# Use smaller subset for testing due to o3 rate limits and cost
test_examples = examples[:100]  # Adjust size as needed

print(f"Evaluating on {len(test_examples)} examples")

# Reduce threads due to o3 rate limits
evaluate = Evaluate(
    devset=test_examples, 
    metric=eval_metric, 
    num_threads=1,  # Lower for o3 models
    display_progress=True, 
    display_table=10, 
    return_outputs=True, 
    return_all_scores=True
)

results = evaluate(o3_sentiment)

# Save results
items = []
for sample in results[1]:
    item = {
        'text': sample[0]['text'],
        'label': sample[0]['label'],
        'pred': extract_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']  # Save full reasoning
    }
    items.append(item)

df_result = pd.DataFrame(data=items)
df_result.to_csv(f'results/sa/{MODEL_NAME}-0shot-sst2.csv', index=False)
print(f"Results saved to results/sa/{MODEL_NAME}-0shot-sst2.csv")
print(f"Accuracy: {results[0]:.3f}")

## Chain-of-Thought with o3 (Advanced Reasoning)

o3 models have built-in reasoning, but we can still use explicit CoT for comparison.

In [ ]:
class CoTO3Sentiment(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(O3Sentiment)

    def forward(self, text):
        return self.prog(text=text)

In [ ]:
cot_o3_sentiment = CoTO3Sentiment()
pred = cot_o3_sentiment(text=example.text)
print("\nQUESTION:\n")
print(example.text)
print("\nPREDICTION:\n")
print(pred)

In [ ]:
lm.inspect_history()

In [ ]:
# Evaluate CoT version (optional - since o3 has built-in reasoning)
evaluate_cot = Evaluate(devset=test_examples[:50], metric=eval_metric, num_threads=1, display_progress=True, display_table=5, return_outputs=True, return_all_scores=True)
results_cot = evaluate_cot(cot_o3_sentiment)

items_cot = []
for sample in results_cot[1]:
    item = {
        'text': sample[0]['text'],
        'label': sample[0]['label'],
        'pred': extract_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']
    }
    items_cot.append(item)

df_result_cot = pd.DataFrame(data=items_cot)
df_result_cot.to_csv(f'{MODEL_NAME}-0shot-cot-sst2.csv', index=False)
print(f"CoT Accuracy: {results_cot[0]:.3f}")

# Evaluate by modification

## Without label change

In [ ]:
def evaluate_modified_set(ds, program, max_samples=50):
    """Evaluate on modified dataset with sample limit."""
    # Limit samples due to o3 cost and rate limits
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "text": remove_space(r['modified_text']), 
                      "original_text": remove_space(r['original_text']),
                      "label": int(r['label']),
                      "modified_label": int(r['label'])
                    }).with_inputs("text") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Recreate classes for consistency
class O3Sentiment(dspy.Signature):
    """Classify sentiment of the given text. Answer with 1 for positive, 0 for negative."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class O3SentimentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Sentiment)

    def forward(self, text):
        return self.prog(text=text)
        
o3_sentiment = O3SentimentModule()

In [ ]:
# Configure o3 model and load original predictions
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

# Load original predictions for comparison
original_pred_file = f'results/sa/{MODEL_NAME}-0shot-sst2.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file, index_col=False)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
    print(f"Loaded original predictions from {original_pred_file}")
else:
    print(f"Original predictions file not found: {original_pred_file}")
    print("Please run the original evaluation first")
    original_pred_ds = None

# Get modification files (subset for testing)
json_files = glob.glob('../data/modified_data/sa/*_100.json')
test_modifications = ['typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json']
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"Testing modifications: {[f.split('/')[-1] for f in json_files]}")

for json_file in json_files:
    print(f"\nProcessing: {json_file}")
    if 'grammatical_role' in json_file or 'negation' in json_file:
        print("Skipping complex modification for now")
        continue
        
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    results = evaluate_modified_set(data, o3_sentiment, max_samples=25)
    
    # Convert results to dataframe
    items = []
    for sample in results[1]:
        item = {}
        sentence = sample[0]['text']
        label = sample[0]['label']
        pred = sample[1]['label']
        item['text'] = sentence
        item['modified_label'] = label
        pred_clean = extract_prediction(pred)
        item['modified_pred'] = pred_clean
        
        original_text = sample[0]['original_text']
        try:
            original_text = original_text.encode('utf-8').decode('unicode-escape')
        except:
            pass
            
        item['original_label'] = sample[0]['label']
        item['original_text'] = original_text
        
        # Find original prediction
        if original_pred_ds is not None:
            matching_rows = original_pred_ds[original_pred_ds['text'] == original_text]
            if not matching_rows.empty:
                item['original_pred'] = matching_rows.iloc[0]['pred']
            else:
                item['original_pred'] = None
        else:
            item['original_pred'] = None
            
        item['raw_output'] = pred
        items.append(item)
    
    df_result = pd.DataFrame(data=items)
    
    # Save results with filename based on input json
    output_filename = f"results/sa/{MODEL_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    print(f"Saved results to: {output_filename}")
    print(f"Accuracy: {results[0]:.3f}")
    
    # Add delay to respect rate limits
    time.sleep(5)

## With label change

In [ ]:
def evaluate_modified_set_with_label_change(ds, program, max_samples=50):
    """Evaluate on modified dataset where labels might change."""
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "text": remove_space(r['modified_text']), 
                      "original_text": remove_space(r['original_text']),
                      "label": int(r['modified_label']) if r.get('modified_label') is not None else int(r['label']),
                      "original_label": int(r['label']),
                      "type": r.get('type', None)
                    }).with_inputs("text") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Test modifications that change labels (e.g., sentiment modifications)
json_files_label_change = glob.glob('../data/modified_data/sa/*_100.json')
label_change_modifications = ['sentiment_100.json']
json_files_label_change = [f for f in json_files_label_change if any(mod in f for mod in label_change_modifications)]

for json_file in json_files_label_change:
    print(f"\nProcessing label-changing modification: {json_file}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    results = evaluate_modified_set_with_label_change(data, o3_sentiment, max_samples=20)
    
    # Convert results to dataframe
    items = []
    for sample in results[1]:
        item = {}
        sentence = sample[0]['text']
        pred = sample[1]['label']
        item['text'] = sentence
        item['modified_label'] = sample[0]['label']
        pred_clean = extract_prediction(pred)
        item['modified_pred'] = pred_clean
        
        original_text = sample[0]['original_text']
        try:
            original_text = original_text.encode('utf-8').decode('unicode-escape')
        except:
            pass
            
        item['original_label'] = sample[0]['original_label']
        item['original_text'] = original_text
        
        # Find original prediction
        if original_pred_ds is not None:
            matching_rows = original_pred_ds[original_pred_ds['text'] == original_text]
            if not matching_rows.empty:
                item['original_pred'] = matching_rows.iloc[0]['pred']
            else:
                item['original_pred'] = None
        else:
            item['original_pred'] = None
            
        item['type'] = sample[0]['type']
        item['raw_output'] = pred
        items.append(item)
    
    df_result = pd.DataFrame(data=items)
    
    # Save results
    output_filename = f"results/sa/{MODEL_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    print(f"Saved results to: {output_filename}")
    print(f"Accuracy: {results[0]:.3f}")
    
    time.sleep(5)

# Aggregate results

In [ ]:
from scipy import stats

In [ ]:
# Aggregate results across all modifications
result_files = glob.glob(f'results/sa/{MODEL_NAME}-0shot-*_100.csv')
aggregated_results = []

print(f"Found {len(result_files)} result files for {MODEL_NAME}")

for file in result_files:
    # Extract modification type from filename
    mod_type = file.split('-')[-1].replace('.csv', '')
    
    # Read results file
    try:
        df = pd.read_csv(file)
        
        # Handle missing columns gracefully
        if 'original_pred' not in df.columns or df['original_pred'].isna().all():
            print(f"Warning: {file} missing original predictions")
            continue
            
        # Calculate accuracies
        original_correct = (df['original_pred'] == df['original_label']).sum()
        modified_correct = (df['modified_pred'] == df['modified_label']).sum()
        total = len(df)

        if total == 0:
            continue
            
        original_acc = original_correct / total
        modified_acc = modified_correct / total
        
        # Calculate the difference
        difference = -round(original_acc - modified_acc, 2)
        
        # Calculate percentage difference
        if original_correct > 0:
            pct_difference = -round((original_correct - modified_correct) / original_correct * 100, 2)
        else:
            pct_difference = 0
        
        # Perform t-test if we have enough data
        try:
            t_stat, p_value = stats.ttest_ind(
                (df['original_pred'] == df['original_label']).astype(float),
                (df['modified_pred'] == df['modified_label']).astype(float)
            )
        except:
            p_value = None
        
        aggregated_results.append({
            'task': 'sentiment_analysis',
            'model': MODEL_NAME,
            'modification': mod_type,
            'original_res': round(original_acc, 3),
            'modified_res': round(modified_acc, 3),
            'difference': difference,
            'pct_difference': pct_difference,
            'p_value': p_value,
            'samples': total
        })
        
    except Exception as e:
        print(f"Error processing {file}: {e}")
        continue

# Create final results dataframe
if aggregated_results:
    results_df = pd.DataFrame(aggregated_results)
    
    # Sort the results
    modification_order = ['temporal_bias_100', 'geographical_bias_100', 'length_bias_100', 
                         'typo_bias_100', 'capitalization_100', 'punctuation_100', 
                         'derivation_100', 'compound_word_100', 'active_to_passive_100',
                         'grammatical_role_100', 'coordinating_conjunction_100', 
                         'concept_replacement_100', 'negation_100', 'discourse_100',
                         'sentiment_100', 'casual_100', 'dialectal_100']
    
    # Only use modifications that exist in our results
    existing_mods = results_df['modification'].unique()
    modification_order = [mod for mod in modification_order if mod in existing_mods]
    
    results_df['modification'] = pd.Categorical(results_df['modification'], categories=modification_order, ordered=True)
    results_df = results_df.sort_values(by='modification')

    # Calculate averages across all modifications
    avg_original = results_df['original_res'].mean()
    avg_modified = results_df['modified_res'].mean()
    avg_difference = avg_original - avg_modified
    avg_pct_difference = results_df['pct_difference'].mean()

    # Add averages as a new row
    avg_row = {
        'task': 'sentiment_analysis',
        'model': MODEL_NAME,
        'modification': 'average',
        'original_res': round(avg_original, 3),
        'modified_res': round(avg_modified, 3),
        'difference': -round(avg_difference, 3),
        'pct_difference': round(avg_pct_difference, 2),
        'p_value': None,
        'samples': results_df['samples'].sum()
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)

    print(f"\n{MODEL_NAME} Results Summary:")
    print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])

    # Save aggregated results
    results_df.to_csv(f'results/sa/{MODEL_NAME}-DP.csv', index=False)
    print(f"\nAggregated results saved to: results/sa/{MODEL_NAME}-DP.csv")

    # Apply styling to highlight performance drops
    def highlight_drops_and_significance(row):
        colors = [''] * len(row)
        if row['original_res'] > row['modified_res']:
            colors = ['background-color: red'] * len(row)
            # If p-value < 0.05, add bold text
            if 'p_value' in row and row['p_value'] is not None and row['p_value'] < 0.05:
                colors = ['background-color: red; font-weight: bold'] * len(row)
        return colors

    styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
    display(styled_df)
    
else:
    print("No results found to aggregate")

## Model Comparison and Analysis

In [ ]:
# Compare with other models if available
comparison_files = {
    'GPT-4o': 'results/sa/gpt4o-0shot-sst2.csv',
    'Claude-3.5': 'results/sa/claude-3-5-sonnet-0shot-sst2.csv',
    'o1-preview': 'results/sa/o1-preview-0shot-sst2.csv',
    MODEL_NAME: f'results/sa/{MODEL_NAME}-0shot-sst2.csv'
}

model_accuracies = {}
for model_name, file_path in comparison_files.items():
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            # Handle different column names
            pred_col = 'pred' if 'pred' in df.columns else 'prediction'
            if pred_col in df.columns and 'label' in df.columns:
                accuracy = (df[pred_col] == df['label']).mean()
                model_accuracies[model_name] = accuracy
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

# Display comparison
if model_accuracies:
    comparison_df = pd.DataFrame([
        {'Model': model, 'Accuracy': acc, 'Performance': f"{acc:.1%}"} 
        for model, acc in model_accuracies.items()
    ])
    comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
    
    print("\nModel Comparison on SST-2:")
    print(comparison_df)

    # Highlight o3 performance
    o3_performance = model_accuracies.get(MODEL_NAME, 0)
    print(f"\n{MODEL_NAME} Accuracy: {o3_performance:.3f} ({o3_performance:.1%})")
    
    if len(model_accuracies) > 1:
        other_models = [acc for model, acc in model_accuracies.items() if model != MODEL_NAME]
        if other_models:
            avg_others = sum(other_models) / len(other_models)
            improvement = o3_performance - avg_others
            print(f"Average of other models: {avg_others:.3f} ({avg_others:.1%})")
            print(f"Performance difference: {improvement:+.3f} ({improvement:+.1%})")

    # Style the dataframe
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]

    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No model comparison data available")

## o3 Reasoning Analysis

In [ ]:
# Analyze o3 reasoning quality (if raw outputs are available)
if 'raw_output' in df_result.columns and not df_result.empty:
    print("Sample o3 Reasoning outputs:")
    print("=" * 50)
    
    for i, (idx, row) in enumerate(df_result.head(3).iterrows()):
        print(f"\nExample {i+1}:")
        print(f"Text: {row['text'][:100]}{'...' if len(row['text']) > 100 else ''}")
        print(f"True Label: {row.get('label', row.get('modified_label', 'N/A'))}")
        print(f"Prediction: {row.get('pred', row.get('modified_pred', 'N/A'))}")
        print(f"Reasoning: {row['raw_output'][:500]}{'...' if len(str(row['raw_output'])) > 500 else ''}")
        print("-" * 40)

# Summary statistics
print(f"\n{MODEL_NAME} Evaluation Summary:")
print("=" * 50)

if 'results' in locals():
    print(f"Base accuracy on SST-2: {results[0]:.3f} ({results[0]:.1%})")

if 'aggregated_results' in locals() and aggregated_results:
    avg_robustness = sum([r['difference'] for r in aggregated_results if r['difference'] is not None]) / len([r for r in aggregated_results if r['difference'] is not None])
    print(f"Average robustness impact: {avg_robustness:+.3f}")
    print(f"Modifications tested: {len(aggregated_results)}")

print(f"\nKey insights with {MODEL_NAME}:")
print(f"- Advanced reasoning model with enhanced chain-of-thought capabilities")
print(f"- Detailed reasoning traces help understand decision process")
print(f"- Performance on linguistic robustness varies by modification type")
print(f"- Higher cost but potentially more reliable for complex reasoning tasks")

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Sentiment Analysis Evaluation with {MODEL_NAME} Complete!")
print(f"{'='*60}")
print(f"Files saved in results/sa/ with prefix '{MODEL_NAME}-'")
print(f"\nNext steps:")
print(f"1. Review reasoning traces in the raw_output column")
print(f"2. Compare robustness with other models")
print(f"3. Analyze which linguistic modifications most challenge {MODEL_NAME}")
print(f"4. Consider cost-benefit for production use")